# Microsoft 에이전트 프레임워크 — Azure OpenAI (응답 API)

이 코드 샘플에서는 **Microsoft 에이전트 프레임워크(MAF)** 를 사용하여 **Responses API** 를 사용하는 **Azure OpenAI** 기반의 간단한 에이전트를 만드는 방법을 보여줍니다.

> **마이그레이션 참고:** 이 샘플은 이전에 GitHub 모델과 함께 Semantic Kernel을 사용했습니다. 이제 Microsoft 에이전트 프레임워크로 마이그레이션되었으며, GitHub 모델(사용 중단 및 2026년 7월 폐기 예정)은 Responses API를 지원하는 Azure OpenAI로 대체되었습니다. MAF의 `OpenAIChatClient`는 Azure OpenAI의 안정적인 `/openai/v1/` 엔드포인트를 대상으로 하며 기본적으로 Responses API를 사용합니다.

이 샘플의 목적은 에이전틱 패턴을 구현할 때 이후 추가 코드 샘플에서 적용할 단계를 시연하는 것입니다.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## 필요한 Python 패키지 가져오기


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## 도구 정의하기

Microsoft Agent Framework에서 <strong>도구</strong>는 `@tool`로 장식된 일반 Python 함수로, 에이전트가 호출할 수 있습니다. 아래는 이전 목적지를 반복하지 않고 랜덤한 휴가 목적지를 반환하는 도구를 정의한 예입니다.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## 에이전트 생성하기

여기서 `TravelAgent`라는 이름의 에이전트를 생성합니다.

이 예제에서는 매우 기본적인 지침을 사용합니다. 에이전트의 행동이 어떻게 변하는지 관찰하려면 이 지침을 자유롭게 수정해 보세요.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## 에이전트 실행하기

이제 에이전트를 실행할 수 있습니다. 에이전트가 대화 내용을 기억하도록 `AgentSession`을 생성한 후, 두 개의 `user_inputs`를 보냅니다. 첫 번째는 여행을 요청하는 것이고, 두 번째는 사용자가 제안이 마음에 들지 않는다며 다른 제안을 요청하는 것입니다 — 에이전트는 세션 기록과 `get_random_destination` 도구를 사용하여 응답합니다.

에이전트가 다르게 반응하는 방식을 관찰하기 위해 이 메시지들을 수정할 수 있습니다. 응답은 토큰 단위로 <strong>스트리밍</strong>됩니다.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
